# Exercise 1: turn the reviews into checked records

**In pairs, 40 minutes in class; the last step at home.** You write a small
module, `extract.py`, that turns each of twenty reviews into a record the
product team can trust: the right shape, checked against two rules, re-prompted
when it fails, retried only when retrying can help. This notebook runs and
checks each piece as you write it.

**How it works:**
- Open **`extract.py`** next to this notebook, in your editor or in Jupyter.
  Each `STEP` there has comments saying exactly what to write. Write the code
  under them and **save the file**: the notebook picks up your change.
- Cells starting with `# GIVEN` run as they are. The other code cells hold only
  comments: write the code under them.
- **Cost:** about $0.05 on Claude Haiku 4.5 for the whole notebook.

In [ ]:
# GIVEN: run this cell as it is.
# It reloads extract.py each time you save it, loads the key from .env, creates the
# client, and loads the 20 reviews and the 20 records a person checked.
%load_ext autoreload
%autoreload 2
import csv
import datetime
import json

import anthropic
from dotenv import load_dotenv
from pydantic import ValidationError

import extract

load_dotenv()
client = anthropic.Anthropic()

with open("data/reviews.json", encoding="utf-8") as f:
    REVIEWS = json.load(f)["reviews"]
with open("data/expected.json", encoding="utf-8") as f:
    EXPECTED = {r["review_id"]: r for r in json.load(f)["records"]}

PRICE_IN, PRICE_OUT = 1.00, 5.00   # $ per million tokens, Claude Haiku 4.5 (Anthropic, 2 Oct 2026)


def cost(tokens_in, tokens_out):
    return (tokens_in * PRICE_IN + tokens_out * PRICE_OUT) / 1e6


print("run on", datetime.date.today(), "| anthropic", anthropic.__version__, "|", len(REVIEWS), "reviews")

## 1. The shape of a record (6 minutes)

In `extract.py`, write **STEP 1**: `AspectMention` and `ReviewRecord`. Then run
the check below.

In [ ]:
# GIVEN: checks your models. A valid record loads; a sentiment of "great" is refused.
good = ('{"language": "en", "sentiment": "positive", "aspects": [{"aspect": "scent", '
        '"polarity": "positive", "evidence": "Smells good"}], "defect_mentioned": false, '
        '"would_recommend": null}')
print(extract.ReviewRecord.model_validate_json(good))
try:
    extract.ReviewRecord.model_validate_json(good.replace('"sentiment": "positive"', '"sentiment": "great"'))
    print("NOT refused: check the sentiment field")
except ValidationError as e:
    print("refused, as it should be:", e.errors()[0]["loc"], e.errors()[0]["type"])
print("fields:", list(extract.ReviewRecord.model_json_schema()["properties"]))

## 2. One request, in the record's shape (8 minutes)

In `extract.py`, write **STEP 2**: `SCHEMA` and `ask(client, messages)`. Then
try it here on three reviews.

In [ ]:
# Try ask on three reviews: REVIEWS[0], REVIEWS[1] and REVIEWS[16].
# For each: build messages = [{"role": "user", "content": extract.user_message(review)}],
# call extract.ask(client, messages), load the text with extract.ReviewRecord.model_validate_json,
# and print the review id, the record, and the usage's input and output tokens.


## 3. The rules a schema cannot express (8 minutes)

In `extract.py`, write **STEP 3**: `CheckedAspect` and `CheckedRecord`. The
check below feeds them three hand-made records.

In [ ]:
# GIVEN: three hand-made records for review 143272 (the pearl headband, in Spanish).
review = next(r for r in REVIEWS if r["review_id"] == 143272)
base = {"language": "es", "sentiment": "positive", "defect_mentioned": False, "would_recommend": None}
cases = {
    "good": [{"aspect": "quality", "polarity": "mixed", "evidence": "de buena calidad"}],
    "an aspect twice": [{"aspect": "quality", "polarity": "positive", "evidence": "de buena calidad"},
                        {"aspect": "quality", "polarity": "negative", "evidence": "El forro se ve un poco frágil"}],
    "a quote not in the review": [{"aspect": "quality", "polarity": "positive", "evidence": "excelente calidad"}],
}
for name, aspects in cases.items():
    try:
        extract.CheckedRecord.model_validate_json(json.dumps({**base, "aspects": aspects}), context={"review": review})
        print(f"{name}: accepted")
    except ValidationError as e:
        print(f"{name}: refused: {e.errors()[0]['msg']}")

## 4. Validate, re-prompt, give up after two (8 minutes)

In `extract.py`, write **STEP 4**: `extract(client, review, max_retries=2)`.
Then run it on the twenty reviews here.

In [ ]:
# Run extract.extract(client, review) on every review in REVIEWS.
# Keep (review, record, attempts) for each in a list called RESULTS,
# and print one line per review: its id, "ok" or "given up", and the number of attempts.

# Write extracted.csv: extract.FIELDS as the header, one extract.to_row(review, record, attempts) per review.


In [ ]:
# GIVEN: what the loop did, and what it cost.
first = sum(1 for _, rec, att in RESULTS if rec and len(att) == 1)
later = sum(1 for _, rec, att in RESULTS if rec and len(att) > 1)
gave_up = [r["review_id"] for r, rec, _ in RESULTS if rec is None]
tin = sum(a["in"] for _, _, att in RESULTS for a in att)
tout = sum(a["out"] for _, _, att in RESULTS for a in att)
print(f"valid first time: {first} | after a re-prompt: {later} | given up: {len(gave_up)} {gave_up}")
for review, _, attempts in RESULTS:
    for a in attempts:
        if a["error"]:
            print(f"  {review['review_id']}: {a['error'][:110]}")
print(f"{tin} in + {tout} out tokens: ${cost(tin, tout):.4f}, so ${cost(tin, tout) / len(RESULTS) * 1000:.2f} per 1,000 reviews")

## 5. One retry policy, yours (5 minutes)

In `extract.py`, write **STEP 5**: `transient(e)`, and the policy wrapped around
`ask`. The check below sends `ask` to a **fake API** that always answers with
the status you choose, so it costs nothing.

In [ ]:
# GIVEN: a fake API that answers every request with one status code, and counts the requests.
import httpx2   # the HTTP library the anthropic SDK uses; installed with it


def fake_client(status):
    calls = []

    def handler(request):
        calls.append(request)
        return httpx2.Response(status, json={"type": "error", "error": {"type": "fake", "message": "fake"}})
    fake = anthropic.Anthropic(api_key="fake", max_retries=0,
                               http_client=httpx2.Client(transport=httpx2.MockTransport(handler)))
    return fake, calls


for status in (529, 400):
    fake, calls = fake_client(status)
    try:
        extract.ask(fake, [{"role": "user", "content": "hi"}])
    except anthropic.APIStatusError as e:
        print(f"the API answers {status}: {type(e).__name__} after {len(calls)} request(s)")

In [ ]:
# Your policy now retries; the SDK must not as well. Replace the client with one created with max_retries=0.


## 6. Compare with records a person checked (5 minutes)

In [ ]:
# GIVEN: your records against the 20 a person checked, field by field.
FIELDS = ["language", "sentiment", "defect_mentioned", "would_recommend"]
agree = {f: 0 for f in FIELDS + ["aspects (same set)"]}
differ = {f: [] for f in agree}
for review, record, _ in RESULTS:
    exp = EXPECTED[review["review_id"]]
    if record is None:
        for f in agree:
            differ[f].append(review["review_id"])
        continue
    for f in FIELDS:
        if getattr(record, f) == exp[f]:
            agree[f] += 1
        else:
            differ[f].append(review["review_id"])
    if {a.aspect for a in record.aspects} == {a["aspect"] for a in exp["aspects"]}:
        agree["aspects (same set)"] += 1
    else:
        differ["aspects (same set)"].append(review["review_id"])
for f in agree:
    print(f"{f:<20} {agree[f]:>2}/{len(RESULTS)}   differ: {differ[f]}")

**Your answer (two or three sentences).** Which field disagrees most with the
checked records? Pick one review where it does, read it, and say whether the
model made an error or the field asks for a judgement two people could make
differently. Then: what does your run cost per 1,000 reviews?

## At home: the same code as a script

In `extract.py`, write the **AT HOME** part: `main(src, dst)`. Then, in a
terminal in this folder, with your virtual environment active:

```bash
python extract.py data/reviews.json extracted.csv
```

**Hand in** your executed notebook, `extract.py` and `extracted.csv`, as the
exercise page says.